# Breast Cancer XGBoost (BCSC Risk Estimation Dataset)

2,392,998 real screening mammograms, aggregated into 280,660 rows by exact
combination of risk factors + outcome (each row's `COUNT` column says how
many real women that combination represents - **every step below has to
account for that, or the numbers are wrong**).

Predicts `CANCER` (diagnosis of invasive or in-situ breast cancer within 1
year of the mammogram) from established BCSC risk factors - reproductive
history, breast density, family history, BMI - NOT from the presenting
symptoms `score_breast_cancer_risk()` already covers in Layer 1 (lump,
nipple discharge, skin changes). Deliberately complementary, not a duplicate.

**Female-only dataset.** This is a screening-mammography population, so this
specific trained model only applies to female patients - see the gender
check in `brain/ml_layer.py`'s `predict_ml_risk()`. Layer 1's rule engine
stays open to every gender, this restriction is Layer 2 (this model) only.

Source: BCSC Risk Estimation Dataset, documented at
bcsc-research.org/datasets/rfdataset/dataset. If you publish anything using
this data, BCSC requires citing: Barlow WE et al., "Prospective breast
cancer risk prediction model for women undergoing screening mammography."
J Natl Cancer Inst. 2006;98:1204-1214.

See `brain/ml_layer.py`'s `BREAST_FEATURE_COLUMNS` and
`_breast_patient_to_row()` for exactly how this model's output is consumed -
this notebook's preprocessing has to produce feature columns and a target
encoding that match those exactly. Called out at each step below.

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report
from xgboost import XGBClassifier
import shap

d:\HACKATHONS\Smart-Horizon-2k26\OncoGuards-48hrs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 0: convert the raw BCSC file into a normal CSV

`datasets/risk.txt` is BCSC's raw ASCII export - whitespace-separated, no
header row, one column per risk factor plus `count`. Column order below
comes straight from BCSC's own published SAS `input` statement, not a
guess. Safe to skip this cell if `datasets/breast_cancer_prediction.csv`
already exists (it does, if you already ran this once) - it's here so the
notebook is reproducible from the raw download alone.

In [2]:
COLUMN_NAMES = [
    "menopaus", "agegrp", "density", "race", "Hispanic", "bmi", "agefirst",
    "nrelbc", "brstproc", "lastmamm", "surgmeno", "hrt", "invasive",
    "cancer", "training", "count",
]

raw = pd.read_csv("../datasets/risk.txt", sep=r"\s+", names=COLUMN_NAMES, header=None)
raw.to_csv("../datasets/breast_cancer_prediction.csv", index=False)
print(f"converted {raw.shape[0]} rows -> datasets/breast_cancer_prediction.csv")

converted 280660 rows -> datasets/breast_cancer_prediction.csv


## EDA and Preprocessing

#### Get a general look at the data

In [3]:
# import data
df = pd.read_csv("../datasets/breast_cancer_prediction.csv")

print(f"shape: {df.shape}")
print(df.info())
print(f"columns: {df.columns.tolist()}")

print("first 5 rows:")
display(df.head())

shape: (280660, 16)
<class 'pandas.DataFrame'>
RangeIndex: 280660 entries, 0 to 280659
Data columns (total 16 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   menopaus  280660 non-null  int64
 1   agegrp    280660 non-null  int64
 2   density   280660 non-null  int64
 3   race      280660 non-null  int64
 4   Hispanic  280660 non-null  int64
 5   bmi       280660 non-null  int64
 6   agefirst  280660 non-null  int64
 7   nrelbc    280660 non-null  int64
 8   brstproc  280660 non-null  int64
 9   lastmamm  280660 non-null  int64
 10  surgmeno  280660 non-null  int64
 11  hrt       280660 non-null  int64
 12  invasive  280660 non-null  int64
 13  cancer    280660 non-null  int64
 14  training  280660 non-null  int64
 15  count     280660 non-null  int64
dtypes: int64(16)
memory usage: 34.3 MB
None
columns: ['menopaus', 'agegrp', 'density', 'race', 'Hispanic', 'bmi', 'agefirst', 'nrelbc', 'brstproc', 'lastmamm', 'surgmeno', 'hrt', 'invasive', 'can

,menopaus,agegrp,density,race,Hispanic,bmi,agefirst,nrelbc,brstproc,lastmamm,surgmeno,hrt,invasive,cancer,training,count
0,0,1,1,1,0,1,0,0,0,0,9,9,0,0,1,4
1,0,1,1,1,0,1,0,0,0,9,9,9,0,0,0,2
2,0,1,1,1,0,1,0,0,0,9,9,9,0,0,1,4
3,0,1,1,1,0,1,0,0,1,9,9,9,0,0,1,1
4,0,1,1,1,0,1,0,1,0,0,9,9,0,0,1,1


### Data Preprocessing

Check for duplicate combination-rows

Unlike lung/cervical, a plain `.duplicated()` isn't the right check here -
this dataset is already aggregated, so a "duplicate" would mean the same
risk-factor combination appearing on two separate rows (a real data
problem), not the same woman appearing twice (expected and fine, that's
what `count` is for). Checking every column EXCEPT `count`.

In [4]:
duplicate_combos = df.duplicated(subset=[c for c in df.columns if c != "count"]).sum()
print(f"duplicate risk-factor combinations: {duplicate_combos}")

# Sanity check against BCSC's own published total - if this doesn't match
# 2,392,998, something went wrong in the Step 0 conversion above.
print(f"total real women represented (sum of count): {df['count'].sum()}")

duplicate risk-factor combinations: 0
total real women represented (sum of count): 2392998


Clean up column names

No spaces or mixed punctuation in these headers (unlike cervical's), so a
plain `.upper()` is enough - no regex needed.

In [5]:
df.columns = df.columns.str.upper()
print(df.columns.tolist())

['MENOPAUS', 'AGEGRP', 'DENSITY', 'RACE', 'HISPANIC', 'BMI', 'AGEFIRST', 'NRELBC', 'BRSTPROC', 'LASTMAMM', 'SURGMENO', 'HRT', 'INVASIVE', 'CANCER', 'TRAINING', 'COUNT']


Rename the target column

`CANCER` -> `LEVEL`, matching the naming convention already used for lung
and cervical's target columns. Using `CANCER` (invasive + in-situ) rather
than `INVASIVE` (invasive only) - matches BCSC's own published model, and
catches early in-situ (DCIS) findings too, which matters for a screening
tool meant to catch things early.

In [6]:
df = df.rename(columns={"CANCER": "LEVEL"})
print(df.columns.tolist())

['MENOPAUS', 'AGEGRP', 'DENSITY', 'RACE', 'HISPANIC', 'BMI', 'AGEFIRST', 'NRELBC', 'BRSTPROC', 'LASTMAMM', 'SURGMENO', 'HRT', 'INVASIVE', 'LEVEL', 'TRAINING', 'COUNT']


Drop `INVASIVE` - an outcome column, not a risk-factor input

`INVASIVE` is a finer-grained version of the exact same diagnosis `LEVEL`
is measuring (every invasive case is also a `LEVEL=1` case). Leaving it in
`X` would be leakage, same treatment cervical gave its other diagnostic
test columns.

In [7]:
df = df.drop(columns=["INVASIVE"])
print(f"shape after dropping INVASIVE: {df.shape}")

shape after dropping INVASIVE: (280660, 15)


Check for missing values

Unlike cervical's `"?"` convention, BCSC encodes "unknown" as a real value
(`9`, or `2` for `NRELBC`) directly in the data, not a blank/NaN - so there
shouldn't be any pandas-null values to deal with here at all.

In [8]:
print(f"missing values: {df.isnull().sum().sum()}")

missing values: 0


### Sanity-check correlations before trusting any feature

Same check the other two notebooks already do - look before training, not
after. But a plain `.corr()` would be WRONG here: it would treat a rare
combination (count=1) exactly the same as a common one (count=7295),
distorting the real population relationship. `np.cov`'s `fweights`
parameter respects how many real women each row actually represents.

In [9]:
FEATURE_COLUMNS = [
    "MENOPAUS", "AGEGRP", "DENSITY", "RACE", "HISPANIC", "BMI",
    "AGEFIRST", "NRELBC", "BRSTPROC", "LASTMAMM", "SURGMENO", "HRT",
]

weights = df["COUNT"].to_numpy()
correlations = {}
for col in FEATURE_COLUMNS:
    cov_matrix = np.cov(df[col], df["LEVEL"], fweights=weights)
    correlations[col] = cov_matrix[0, 1] / (cov_matrix[0, 0] ** 0.5 * cov_matrix[1, 1] ** 0.5)

correlations = pd.Series(correlations).sort_values(ascending=False)
print(correlations)

AGEGRP      0.024534
DENSITY     0.006612
BRSTPROC    0.001817
LASTMAMM    0.001387
NRELBC     -0.000868
HISPANIC   -0.001533
BMI        -0.002058
RACE       -0.002142
AGEFIRST   -0.002633
MENOPAUS   -0.003089
SURGMENO   -0.011440
HRT        -0.012799
dtype: float64


`AGEGRP` is the clear standout (~0.025) - expected, age is the single
strongest breast cancer risk factor in the real literature too. Everything
else is small and plausible, real epidemiological data again, not a
synthetic-looking dataset.

Worth flagging: `SURGMENO` and `HRT` both come back slightly NEGATIVE here.
For `HRT` especially, that cuts against the mainstream literature direction
(current hormone therapy is generally associated with INCREASED risk, not
decreased). Could be confounding in this simple marginal check, could be a
real cohort effect - either way, this is exactly why `ml_layer.py`'s
`BREAST_CLINICALLY_DEFENSIBLE` set excludes both, same "don't assume the
label matches conventional wisdom" caution already applied to cervical's
`IUD` field.

### Feature selection and train-test split

Using BCSC's own `TRAINING` column instead of `train_test_split` - this
dataset ships with an official, pre-randomized 75%/25% split specifically
so results are reproducible across different people using the data. Simpler
than a manual split, and it's literally the "correct" way to use this
particular dataset.

In [10]:
# Column order here matches brain/ml_layer.py's BREAST_FEATURE_COLUMNS
# exactly - not strictly required (XGBoost matches by name, not position),
# but keeps training and inference trivially easy to compare side by side.
assert set(FEATURE_COLUMNS) == set(df.columns) - {"LEVEL", "TRAINING", "COUNT"}, \
    "feature columns don't match the dataframe - check for a typo above"

train_df = df[df["TRAINING"] == 1]
test_df = df[df["TRAINING"] == 0]

X_train, y_train, w_train = train_df[FEATURE_COLUMNS], train_df["LEVEL"], train_df["COUNT"]
X_test, y_test, w_test = test_df[FEATURE_COLUMNS], test_df["LEVEL"], test_df["COUNT"]

print(f"train: {X_train.shape} ({w_train.sum():,} real women)")
print(f"test:  {X_test.shape} ({w_test.sum():,} real women)")

train: (180465, 12) (1,795,139 real women)
test:  (100195, 12) (597,859 real women)


#### Check class balance

Weighted by `count` again - a plain `.value_counts()` on the 180K unique
combination-rows would badly understate how rare cancer actually is,
since it would count a rare combo (1 woman) the same as a common one
(thousands of women).

In [11]:
pos_weight = w_train[y_train == 1].sum()
neg_weight = w_train[y_train == 0].sum()
print(f"positive: {pos_weight:,}  negative: {neg_weight:,}")
print(f"positive rate: {pos_weight / (pos_weight + neg_weight):.3%}")

scale_pos_weight = neg_weight / pos_weight
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

positive: 8,767  negative: 1,786,372
positive rate: 0.488%
scale_pos_weight: 203.8


~0.49% positive - even more imbalanced than cervical's 6.4%, but with
~1.8 million real weighted training observations behind it (vs. cervical's
858 rows), there's no shortage of positive examples in absolute terms
(~8,700 of them). **Using `scale_pos_weight` instead of SMOTENC this time**
- oversampling millions of rows to rebalance them would be slow and memory-
heavy for no real benefit at this sample size; `scale_pos_weight` tells
XGBoost's loss function to weight each positive example more heavily
instead, without duplicating any data.

#### Train the XGBoost Model

`sample_weight=w_train` is the other half of handling aggregated data - it
tells XGBoost "this row isn't one observation, it's `count` identical
observations", so the model is trained on the real population distribution
instead of the 180,465 unique combinations as if they were equally common.
Same `max_depth`/`n_estimators` as the other two notebooks, kept consistent
on purpose - deeper hyperparameter tuning is a reasonable next step, just
out of scope here.

In [12]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
)

model.fit(X_train, y_train, sample_weight=w_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


#### Model Evaluation

`sample_weight=w_test` again - without it, the report would silently
evaluate on the 100,195 unique test combinations as if they were equally
likely, instead of the real ~598K women they represent.

In [13]:
preds = model.predict(X_test)
print(classification_report(
    y_test, preds,
    target_names=["No Cancer", "Cancer"],
    sample_weight=w_test,
))

              precision    recall  f1-score   support

   No Cancer       1.00      0.57      0.72  594988.0
      Cancer       0.01      0.65      0.01    2871.0

    accuracy                           0.57  597859.0
   macro avg       0.50      0.61      0.37  597859.0
weighted avg       0.99      0.57      0.72  597859.0



This test set represents ~598K real women (~2,871 real positive cases) -
far more statistically stable ground to evaluate on than cervical's 55-row
test set. Still, at well under 1% positive, expect precision on the
positive class to look modest even for a genuinely useful model - that's
the nature of screening a rare outcome, not a sign the model is broken.

#### SHAP Explainer

Binary model, same shape-check caution as lung/cervical - don't assume.
Using a 5,000-row random sample of the test set rather than the full
~100K rows: SHAP's `TreeExplainer` is exact and reasonably fast, but there's
no reason to wait on the full test set just to get a stable feature-
importance ranking.

In [14]:
X_test_sample = X_test.sample(5000, random_state=42)
y_test_sample = y_test.loc[X_test_sample.index]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_sample)

print(f"type: {type(shap_values)}")
if isinstance(shap_values, list):
    print(f"list of {len(shap_values)} arrays, each shape: {shap_values[0].shape}")
else:
    print(f"shape: {shap_values.shape}")

type: <class 'numpy.ndarray'>
shape: (5000, 12)


#### Look at one patient

Same pattern as the other two notebooks - pick a sampled test-set patient,
compare actual vs. predicted, and pull out that patient's SHAP
contributions.

In [15]:
i = 0  # first patient in the sample

predicted_class = int(model.predict(X_test_sample.iloc[[i]])[0])
actual_class = int(y_test_sample.iloc[i])
labels = ["No Cancer", "Cancer"]

print(X_test_sample.iloc[i])
print(f"\nActual: {labels[actual_class]}, Predicted: {labels[predicted_class]}")

# adjust this line based on what the shape-check cell above actually printed
patient_shap = shap_values[i] if not isinstance(shap_values, list) else shap_values[predicted_class][i]
print(dict(zip(X_test_sample.columns, patient_shap)))

MENOPAUS    1
AGEGRP      8
DENSITY     3
RACE        3
HISPANIC    9
BMI         9
AGEFIRST    9
NRELBC      9
BRSTPROC    9
LASTMAMM    0
SURGMENO    9
HRT         9
Name: 216126, dtype: int64

Actual: No Cancer, Predicted: Cancer
{'MENOPAUS': np.float32(-0.008405563), 'AGEGRP': np.float32(0.36173835), 'DENSITY': np.float32(0.12790179), 'RACE': np.float32(0.05107716), 'HISPANIC': np.float32(-0.056633446), 'BMI': np.float32(-0.043246787), 'AGEFIRST': np.float32(0.028404558), 'NRELBC': np.float32(-0.21019465), 'BRSTPROC': np.float32(-0.055853423), 'LASTMAMM': np.float32(0.0038741727), 'SURGMENO': np.float32(-0.039479535), 'HRT': np.float32(0.028523088)}


More detailed look at SHAP importance (averaged across the sampled test
patients, for one overall ranking)

In [16]:
shap_matrix = shap_values if not isinstance(shap_values, list) else shap_values[1]
mean_abs_shap = np.abs(shap_matrix).mean(axis=0)
importance = sorted(zip(X_test_sample.columns, mean_abs_shap), key=lambda x: -x[1])
for feature, val in importance:
    print(f"{feature}: {val:.3f}")

AGEGRP: 0.401
DENSITY: 0.271
NRELBC: 0.116
BRSTPROC: 0.115
LASTMAMM: 0.088
BMI: 0.076
HRT: 0.073
AGEFIRST: 0.058
RACE: 0.054
HISPANIC: 0.053
SURGMENO: 0.049
MENOPAUS: 0.018


Save the model

Relative path, run this notebook with the project root as the working
directory - matches where `brain/ml_layer.py`'s `MODELS_DIR / "breast_xgb_model.json"` looks for it.

In [17]:
model.save_model("../models/breast_xgb_model.json")
print("saved to models/breast_xgb_model.json")

saved to models/breast_xgb_model.json
